# Data Preparation: Part 7 - Additional ASRS Data

## 1. Import Packages & Define Custom Functions

In [1]:
import pandas as pd
import re

from feature_engine.encoding import OneHotEncoder, RareLabelEncoder
from feature_engine.imputation import CategoricalImputer

import sklearn 
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [2]:
# Using custom function from Michael Albert's class

def summarize_dataframe(df):
    missing_values = pd.concat([pd.DataFrame(df.columns, columns=['Variable Name']), 
                      pd.DataFrame(df.dtypes.values.reshape([-1,1]), columns=['Data Type']),
                      pd.DataFrame(df.isnull().sum().values, columns=['Missing Values']), 
                      pd.DataFrame([df[name].nunique() for name in df.columns], columns=['Unique Values'])], 
                     axis=1).set_index('Variable Name')
    return pd.concat([missing_values, df.describe(include='all').transpose()], axis=1).fillna("")

## 2. Import/Subset Raw Data

In [ ]:
full = pd.read_csv('02_Data_Cleaning/Raw_Input_Files/asrs_addtl_n33723.csv')

In [4]:
subset = full[['acn', 'FlightConditions', 'Light', 'CrewSize', 'Mission', 'PrimaryProblem']]

## 3. Clean Data

### 3.1. Treat Missing Values

In [5]:
# Identify variables with any missing values
data_miss = subset.columns[subset.isnull().any()].tolist()

In [6]:
# List missing categorical columns
cat_miss = subset[data_miss].columns[subset[data_miss].dtypes == 'object'].tolist()
cat_miss

['FlightConditions', 'Light', 'Mission', 'PrimaryProblem']

In [7]:
# Replace categorical columns
imp = CategoricalImputer(imputation_method = 'missing', 
                         fill_value = 'UNKNOWN',
                         variables = cat_miss)

data_cl = imp.fit_transform(subset)

### 3.2. Update Categories

**NOTE:** Using the following variable categories:

- *FlightConditions:* Keeping/making all categorical (IMC, Marginal, Mixed, VMC, UNKNOWN [for missing])
- *Light:* Keeping/making all categorical (Dawn, Daylight, Dusk, Night, UNKNOWN [for missing])
- *CrewSize:* n/a - numerical
- *Mission:* Keeping top (Passenger, Personal, Training, Cargo / Freight / Delivery, UNKNOWN [for missing] -- all else OTHER)
- *PrimaryProblem:* Keeping top (Human Factors, Aircraft, Procedure, Weather, UNKNOWN [for missing] -- all else OTHER)

#### 3.2a. Update 'Mission' var for OTHER category

In [8]:
with pd.option_context('display.max_rows', None):
    display(data_cl[['Mission']].value_counts())

Mission                                        
Passenger                                          18900
Personal                                            4079
Training                                            3937
UNKNOWN                                             3502
Cargo / Freight / Delivery                          1599
Ferry / Re-Positioning                               786
Test Flight / Demonstration                          155
Photo Shoot / Video                                  148
Ambulance                                            105
Utility / Infrastructure                              84
Recreational / Hobbyist (UAS)                         60
Skydiving                                             44
Other unknown                                         39
Tactical                                              36
Surveying / Mapping (UAS)                             35
Agriculture                                           31
Public Safety / Pursuit (UAS)           

In [9]:
enc = RareLabelEncoder(tol=0.04, 
                       variables = 'Mission',
                       replace_with = 'OTHER')

data_enc = enc.fit_transform(data_cl)

In [10]:
with pd.option_context('display.max_rows', None):
    display(data_enc[['Mission']].value_counts())

Mission                   
Passenger                     18900
Personal                       4079
Training                       3937
UNKNOWN                        3502
OTHER                          1706
Cargo / Freight / Delivery     1599
Name: count, dtype: int64

#### 3.2b. Update 'PrimaryProblem' var for OTHER category

In [11]:
with pd.option_context('display.max_rows', None):
    display(data_enc[['PrimaryProblem']].value_counts())

PrimaryProblem                              
Human Factors                                   11657
Aircraft                                        10185
Procedure                                        3623
Ambiguous                                        2478
Weather                                          1152
Environment - Non Weather Related                 969
Airport                                           788
Company Policy                                    605
Chart Or Publication                              534
Airspace Structure                                512
ATC Equipment / Nav Facility / Buildings          436
Equipment / Tooling                               156
Staffing                                          140
Software and Automation                           136
MEL                                               112
UNKNOWN                                            86
Incorrect / Not Installed / Unavailable Part       70
Manuals                              

In [12]:
enc = RareLabelEncoder(tol=0.03, 
                       variables = 'PrimaryProblem',
                       replace_with = 'OTHER')

data_enc = enc.fit_transform(data_enc)

In [13]:
with pd.option_context('display.max_rows', None):
    display(data_enc[['PrimaryProblem']].value_counts())

PrimaryProblem
Human Factors     11657
Aircraft          10185
OTHER              4628
Procedure          3623
Ambiguous          2478
Weather            1152
Name: count, dtype: int64

In [14]:
# Change "Ambiguous" to "OTHER"
data_enc.loc[data_enc['PrimaryProblem'] == "Ambiguous", 'PrimaryProblem'] = "OTHER"

In [15]:
data_enc[['PrimaryProblem']].value_counts()

PrimaryProblem
Human Factors     11657
Aircraft          10185
OTHER              7106
Procedure          3623
Weather            1152
Name: count, dtype: int64

### 3.4. OneHotEncoder

In [16]:
data_enc2 = data_enc.copy()

In [17]:
to_enc = data_enc2.drop(['acn', 'CrewSize'], axis=1)

In [18]:
# Set up OneHotEncoder
enc = OneHotEncoder(sparse_output=False)

# Fit
data_ohe = pd.DataFrame(
    enc.fit_transform(to_enc),
    columns = enc.get_feature_names_out(),
    index = to_enc.index)

In [19]:
# Concat original/initial-cleaned data with new dummies
data_ohe2 = pd.concat([data_cl, data_ohe], axis=1)

In [20]:
with pd.option_context('display.max_rows', None):
    display(summarize_dataframe(data_ohe2))

/tmp/ipykernel_2159/3979652495.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return pd.concat([missing_values, df.describe(include='all').transpose()], axis=1).fillna("")


,Data Type,Missing Values,Unique Values,count,unique,top,freq,mean,std,min,25%,50%,75%,max
acn,int64,0,33723,33723.0,,,,1828811.478783,192110.173434,1507557.0,1673862.5,1806271.0,1981117.5,2296796.0
FlightConditions,object,0,5,33723.0,5,VMC,17077,,,,,,,
Light,object,0,5,33723.0,5,UNKNOWN,15741,,,,,,,
CrewSize,float64,2825,9,30898.0,,,,1.824099,0.494949,1.0,2.0,2.0,2.0,12.0
Mission,object,0,122,33723.0,122,Passenger,18900,,,,,,,
PrimaryProblem,object,0,19,33723.0,19,Human Factors,11657,,,,,,,
FlightConditions_IMC,float64,0,2,33723.0,,,,0.058476,0.234646,0.0,0.0,0.0,0.0,1.0
FlightConditions_Marginal,float64,0,2,33723.0,,,,0.016991,0.129241,0.0,0.0,0.0,0.0,1.0
FlightConditions_Mixed,float64,0,2,33723.0,,,,0.016339,0.126777,0.0,0.0,0.0,0.0,1.0
FlightConditions_UNKNOWN,float64,0,2,33723.0,,,,0.401803,0.49027,0.0,0.0,0.0,1.0,1.0


## 4. Export Data

In [ ]:
# Export
# data_ohe2.to_csv('02_Data_Cleaning/Output_Files/Data 7 - addtl_ASRS clean.csv', index=False)